# Test your fine-tuned T5 locally

Open this notebook in VS Code or Jupyter and select the interpreter
`C:/Users/ADMIN/ai_venv/Scripts/python.exe`. Run the cells from the top.
The existing successful export in `qa_baseline_artifacts/` is selected below.
You can choose another downloaded `hf_export/` folder to test a later model.
The helper source is imported from this local project. Model loading is
offline, and CUDA is used if available, with CPU as the fallback.

Provide a passage and ask questions about it. This model was trained for
passage-based question answering, rather than general chat.

In [ ]:
from pathlib import Path
import sys

VENV_PYTHON = Path(r"C:\Users\ADMIN\ai_venv\Scripts\python.exe")
PROJECT_DIR = None  # optionally Path(r"C:\path\to\Asgm_1") if opened elsewhere
if Path(sys.executable).resolve() != VENV_PYTHON.resolve():
    raise RuntimeError(f"Select this notebook's Python kernel: {VENV_PYTHON}. Current: {sys.executable}")
if PROJECT_DIR is None:
    PROJECT_DIR = next((directory for directory in (Path.cwd(), *Path.cwd().parents)
                        if (directory / "pyproject.toml").is_file() and
                        (directory / "src" / "qa_assignment").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open the notebook from the assignment folder, or set PROJECT_DIR explicitly.")
PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
SOURCE_ROOT = PROJECT_DIR / "src"
if not (SOURCE_ROOT / "qa_assignment" / "inference.py").is_file():
    raise FileNotFoundError(f"Cannot find the local helpers under {SOURCE_ROOT}.")
OUTPUT_ROOT = PROJECT_DIR / "outputs" / "t5_qa_test"
print("Python:", sys.executable)
print("Project:", PROJECT_DIR)

## Import local helpers

The selected venv must already contain the project's Python dependencies.
Setup imports them without installing packages or modifying your environment.
GPU is optional. If you prefer CPU, set `DEVICE="cpu"` below.

In [ ]:
import importlib

if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()
import qa_assignment
if Path(qa_assignment.__file__).resolve().parent != SOURCE_ROOT / "qa_assignment":
    raise RuntimeError("Helpers were imported from another checkout. Restart the kernel and rerun setup.")
import torch
from qa_assignment.inference import T5Answerer, find_t5_exports

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Inference device:", DEVICE)

## Load your saved model

The default `MODEL_DIR` points to the successful run already in this project.
Relative paths resolve against `PROJECT_DIR`; an absolute Windows path also
works. Select the `hf_export/` folder containing the model and tokenizer.
An empty setting discovers exports under the local artifact/output folders;
if there are multiple exports, choose one explicitly.
The model and tokenizer load entirely from disk, without fetching new weights.
Run this cell once; subsequent question cells reuse the loaded model.

In [ ]:
from qa_assignment.inference import T5Answerer, find_t5_exports

MODEL_DIR = "qa_baseline_artifacts/rnn_lstm_attn_t5/checkpoints/t5_small/seed_42/hf_export"
exports = find_t5_exports([PROJECT_DIR / "qa_baseline_artifacts",
                           PROJECT_DIR / "checkpoints", PROJECT_DIR / "outputs"])
print("Available T5 exports:")
for index, path in enumerate(exports):
    print(index, path)
if not MODEL_DIR.strip():
    if len(exports) != 1:
        raise ValueError("Set MODEL_DIR to the hf_export folder from your downloaded model outputs.")
    MODEL_DIR = str(exports[0])
model_path = Path(MODEL_DIR).expanduser()
if not model_path.is_absolute():
    model_path = PROJECT_DIR / model_path
MODEL_DIR = str(model_path.resolve())
qa = T5Answerer.from_export(MODEL_DIR, device=DEVICE)
print("Loaded:", MODEL_DIR)
print("Question + passage limit:", qa.max_input_length, "tokens")

## Your passage and question

Replace the passage and question below, then rerun these cells to ask more.
The complete question plus passage must fit the saved input limit (normally
512 tokens); overly long inputs are rejected so the answer's context is not
silently discarded. Questions with no answer in the passage can still receive
an incorrect answer: this SQuAD 1.1 model has no trained abstention behavior.

In [ ]:
PASSAGE = (
    "Super Bowl 50 was played on February 7, 2016, at Levi's Stadium in "
    "Santa Clara, California. The Denver Broncos defeated the Carolina "
    "Panthers by a score of 24-10. Von Miller was named the game's MVP."
)
QUESTION = "Which team won Super Bowl 50?"
print("Answer:", qa.answer(QUESTION, PASSAGE) or "<empty answer>")

## Ask several questions about the same passage

Edit this list and rerun the cell. The model stays loaded. Each question is
answered independently from the passage; prior answers are not chat history.

In [ ]:
QUESTIONS = [
    "Where was Super Bowl 50 played?",
    "Who was named the game's MVP?",
    "When was Super Bowl 50 played?",
]
for result in qa.answer_many(QUESTIONS, PASSAGE):
    print("Question:", result["question"])
    print("Answer:", result["answer"] or "<empty answer>")

## Optional typed questions and saved answers

For a manual session, enable the input loop below. Submit an empty question
to exit. It defaults off so running every cell does not wait for typed input.
Answers from either interface can be saved locally.

In [ ]:
RUN_INTERACTIVE_CHAT = False
if RUN_INTERACTIVE_CHAT:
    qa.interactive(PASSAGE)

In [ ]:
from qa_assignment.utils import write_json

answer_log = OUTPUT_ROOT / "answers.json"
write_json(answer_log, {"model_dir": MODEL_DIR, "answers": qa.history})
print("Saved answers:", answer_log)